Hello

# Mycobacterium tuberculosis (TaxID 1773) – BioAssay Comparison

Objective:
Compare BioAssays obtained from:
1. **PubChem Web UI export** for TaxID 1773
2. **Aid2Taxid.tsv explicit taxonomy mapping**

We identify assays that appear in the Web UI download but *not* in Aid2Taxid,
and inspect metadata to understand the differences.

## 0. Setup

In [1]:
import pandas as pd
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
DATA_RAW = NOTEBOOK_DIR.parent / "data" / "raw"
DATA_PROCESSED = NOTEBOOK_DIR.parent / "data" / "processed"

NOTEBOOK_DIR, DATA_RAW

(PosixPath('/Users/maria/Documents/Ersilia/PubChem/pubchem-antimicrobial-tasks/notebooks'),
 PosixPath('/Users/maria/Documents/Ersilia/PubChem/pubchem-antimicrobial-tasks/data/raw'))

## 1. Load UI-exported BioAssays for MTB

File downloaded manually from:

**PubChem → Taxonomy → Mycobacterium tuberculosis (TaxID 1773) → Actions → BioAssays → Summary**

In [2]:
df_ui = pd.read_csv(DATA_RAW / "pubchem_taxid_1773_bioassay.csv")

df_ui.head()

,aid,cnt,aidtype,aidname,aiddesc,aidcomment,aidsrcid,aidsrcname,aidextid,depdate,...,taxids,cellids,targettaxid,activecnt,anatomyid,anatomy,dois,pmcids,pclids,citations
0,1671161,86590.0,Confirmatory,Phenotypic growth assay for Mycobacterium tube...,Phenotypic growth assay for Mycobacterium tube...,Compounds with activity <= 10uM or explicitly ...,43,ChEMBL,CHEMBL4649948,20210802,...,1773,NaN,NaN,68.0,NaN,NaN,NaN,NaN,NaN,NaN
1,1671162,86576.0,Confirmatory,Phenotypic growth assay for Mycobacterium tube...,Phenotypic growth assay for Mycobacterium tube...,Compounds with activity <= 10uM or explicitly ...,43,ChEMBL,CHEMBL4649949,20210802,...,1773,NaN,NaN,90.0,NaN,NaN,NaN,NaN,NaN,NaN
2,1671184,68614.0,Confirmatory,Cornell M. t heat shock assay,Cornell M. t heat shock assay,Compounds with activity <= 10uM or explicitly ...,43,ChEMBL,CHEMBL4649971,20210802,...,1773,NaN,1773.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN
3,1671185,68611.0,Confirmatory,in vitro assay to detect presence of AMC direc...,in vitro assay to detect presence of AMC direc...,Compounds with activity <= 10uM or explicitly ...,43,ChEMBL,CHEMBL4649972,20210802,...,1773|83332,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN
4,1671154,66942.0,Confirmatory,"Biomol Green assay for MtCoaBC inhibitor, OD65...","Biomol Green assay for MtCoaBC inhibitor, OD65...",Compounds with activity <= 10uM or explicitly ...,43,ChEMBL,CHEMBL4649941,20210802,...,1773|83332,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Load Aid2Taxid.tsv mapping file

In [16]:
df_map = pd.read_csv(DATA_RAW / "Aid2Taxid.tsv", sep="\t")

df_map.head()

,AID,TaxID
0,1398771,10090
1,1398772,10090
2,1398788,9606
3,1398789,9606
4,1398790,9606


## 3. Define the MTB TaxID

We only examine **TaxID = 1773**.

In [3]:
mtb_taxid = "1773"

## 4. Extract AIDs explicitly linked to MTB in Aid2Taxid

In [5]:
df_map_mtb = df_map[df_map["TaxID"].astype(str) == mtb_taxid]

aid2tax_aids = set(df_map_mtb["AID"].astype(int))
len(aid2tax_aids)

6278

## 5. Extract UI-linked AIDs

In [4]:
ui_aids = set(df_ui["aid"].astype(int))
len(ui_aids)

11199

## 6. Compare the two sets

In [7]:
missing_from_aid2tax = ui_aids - aid2tax_aids
extra_in_aid2tax = aid2tax_aids - ui_aids

summary = pd.DataFrame({
    "Metric": ["UI AIDs", "Aid2Taxid AIDs", "Missing (UI→Aid2Taxid)", "Unexpected (Aid2Taxid→UI)"],
    "Count": [len(ui_aids), len(aid2tax_aids), len(missing_from_aid2tax), len(extra_in_aid2tax)]
})

summary

,Metric,Count
0,UI AIDs,11199
1,Aid2Taxid AIDs,6278
2,Missing (UI→Aid2Taxid),4922
3,Unexpected (Aid2Taxid→UI),1


## 7. Inspect Missing AIDs (in UI but *not* in Aid2Taxid)

In [8]:
df_missing = df_ui[df_ui["aid"].isin(missing_from_aid2tax)].copy()

df_missing.head()

,aid,cnt,aidtype,aidname,aiddesc,aidcomment,aidsrcid,aidsrcname,aidextid,depdate,...,taxids,cellids,targettaxid,activecnt,anatomyid,anatomy,dois,pmcids,pclids,citations
10,977608,4302.0,Literature-derived,Experimentally measured binding affinity data ...,This data entry provides a collection of exper...,NaN,36,Shanghai Institute of Organic Chemistry,PDBbind-IC50 for protein-ligand complexes,20140506,...,34|210|274|287|512|536|550|562|573|615|666|727...,NaN,NaN,4302.0,NaN,NaN,10.1002/(sici)1097-0282(20000415)53:5<434::aid...,PMC11113908|PMC1157103|PMC1222325|PMC1241520|P...,159245|188532|189137|193886|208525|225116|2355...,"Hassan C, Chabrol E, Jahn L, Kester MGD, de Ru..."
11,977611,3916.0,Literature-derived,Experimentally measured binding affinity data ...,This data entry provides a collection of exper...,NaN,36,Shanghai Institute of Organic Chemistry,PDBbind-Kd for protein-ligand complexes,20140506,...,160|197|210|235|266|271|274|285|287|292|294|30...,NaN,NaN,3916.0,NaN,NaN,10.1002/(sici)1097-0134(199709)29:1<113::aid-p...,PMC1064020|PMC11114729|PMC1134746|PMC1142576|P...,152137|182245|182284|189113|208525|222463|2345...,"Shukla PK, Gautam L, Sinha M, Kaur P, Sharma S..."
12,977610,3598.0,Literature-derived,Experimentally measured binding affinity data ...,This data entry provides a collection of exper...,NaN,36,Shanghai Institute of Organic Chemistry,PDBbind-Ki for protein-ligand complexes,20140506,...,11|69|173|197|210|232|238|271|274|287|292|293|...,NaN,NaN,3598.0,NaN,NaN,10.1002/(sici)1097-0134(19990601)35:4<425::aid...,PMC10316677|PMC11114634|PMC1134741|PMC1169963|...,49317|189199|226130|233956|235576|241483|24546...,"Kick E, Martin R, Xie Y, Flatt B, Schweiger E,..."
13,1811,3073.0,Literature-derived,Experimentally measured binding affinity data ...,This data entry provides a collection of exper...,NaN,36,Shanghai Institute of Organic Chemistry,PDBBind Data 1,20090608,...,69|160|197|210|238|266|271|274|285|287|294|303...,NaN,NaN,3073.0,NaN,NaN,10.1002/(sici)1097-0134(19990601)35:4<425::aid...,PMC1064020|PMC1134741|PMC1134746|PMC1142596|PM...,49317|152137|182245|182284|188532|189113|18913...,"Smaill JB, Baker EN, Booth RJ, Bridges AJ, Dic..."
14,1671503,400.0,Other,Inhibition of mycobacterial growth using the r...,Inhibition of mycobacterial growth using the r...,Journal: Lancet Microbe_||_Year: 2021_||_DOI: ...,43,ChEMBL,CHEMBL4665544,20220318,...,1773,NaN,1773.0,57.0,NaN,NaN,NaN,NaN,NaN,NaN


### 7.1 Where do the missing assays come from?

In [9]:
df_missing["aidsrcname"].value_counts()

aidsrcname
ChEMBL                                     4918
Shanghai Institute of Organic Chemistry       4
Name: count, dtype: int64

In [10]:
df_missing["aidtype"].value_counts()

aidtype
Literature-derived    3410
Confirmatory          1505
Other                    7
Name: count, dtype: int64

### 7.2 Evaluate TaxIDs inside the `taxids` column

In [17]:
def extract_taxids(taxid_field):
    """Return list of TaxIDs (strings) from the pipe-separated taxid field."""
    if pd.isna(taxid_field):
        return []
    return str(taxid_field).split("|")

In [ ]:
df_missing["parsed_taxids"] = df_missing["taxids"].apply(extract_taxids)

df_missing["has_mtb"] = df_missing["parsed_taxids"].apply(
    lambda lst: mtb_taxid in lst
)

df_missing["has_mtb"].value_counts()

has_mtb
True    4922
Name: count, dtype: int64

### 7.3 Inspect AIDs that DO contain the MTB TaxID

In [12]:
df_missing_with_mtb = df_missing[df_missing["has_mtb"]]

df_missing_with_mtb[["aid", "aidsrcname", "aidname", "taxids", "parsed_taxids"]].head(20)

,aid,aidsrcname,aidname,taxids,parsed_taxids
10,977608,Shanghai Institute of Organic Chemistry,Experimentally measured binding affinity data ...,34|210|274|287|512|536|550|562|573|615|666|727...,"[34, 210, 274, 287, 512, 536, 550, 562, 573, 6..."
11,977611,Shanghai Institute of Organic Chemistry,Experimentally measured binding affinity data ...,160|197|210|235|266|271|274|285|287|292|294|30...,"[160, 197, 210, 235, 266, 271, 274, 285, 287, ..."
12,977610,Shanghai Institute of Organic Chemistry,Experimentally measured binding affinity data ...,11|69|173|197|210|232|238|271|274|287|292|293|...,"[11, 69, 173, 197, 210, 232, 238, 271, 274, 28..."
13,1811,Shanghai Institute of Organic Chemistry,Experimentally measured binding affinity data ...,69|160|197|210|238|266|271|274|285|287|294|303...,"[69, 160, 197, 210, 238, 266, 271, 274, 285, 2..."
14,1671503,ChEMBL,Inhibition of mycobacterial growth using the r...,1773,[1773]
17,1992963,ChEMBL,Antimycobacterial activity against Mycobacteri...,1773,[1773]
18,1992964,ChEMBL,Antimycobacterial activity against Mycobacteri...,1773,[1773]
22,661399,ChEMBL,Antimycobacterial activity against Mycobacteri...,1773,[1773]
23,541701,ChEMBL,Antitubercular activity against Mycobacterium ...,1773,[1773]
24,541702,ChEMBL,Antitubercular activity against Mycobacterium ...,1773,[1773]


### 7.4 Inspect AIDs that do NOT contain the MTB TaxID

In [13]:
df_missing_no_mtb = df_missing[~df_missing["has_mtb"]]

df_missing_no_mtb[["aid", "aidsrcname", "aidname", "parsed_taxids"]].head(20)

,aid,aidsrcname,aidname,parsed_taxids


### 7.5 Are missing assays mapped to ANY of our 15 pathogens?

In [14]:
# Load pathogen taxonomy table
tax_table = pd.read_csv(DATA_PROCESSED / "taxonomy_table.csv")

pathogen_tax_map = (
    tax_table.groupby("Pathogen")["Taxonomy_ID"]
    .apply(lambda x: set(map(str, x)))
    .to_dict()
)

def match_pathogens(tid_list):
    matches = []
    for pathogen, ids in pathogen_tax_map.items():
        if any(t in tid_list for t in ids):
            matches.append(pathogen)
    return matches

df_missing_no_mtb["other_pathogen_hits"] = df_missing_no_mtb["parsed_taxids"].apply(match_pathogens)

df_missing_no_mtb[["aid", "parsed_taxids", "other_pathogen_hits"]].head(20)

,aid,parsed_taxids,other_pathogen_hits


### 7.6 Missing values in TaxIDs

In [15]:
df_missing["taxids"].isna().sum()

0

## 8. Summary of missing categories

In [16]:
summary_missing = pd.DataFrame({
    "Category": [
        "Missing AIDs (UI→Aid2Taxid)",
        "Contain MTB TaxID",
        "Match Other Pathogens",
        "Match No Known Pathogens",
        "No TaxID Present"
    ],
    "Count": [
        len(df_missing),
        df_missing["has_mtb"].sum(),
        sum(df_missing_no_mtb["other_pathogen_hits"].apply(len) > 0),
        sum(df_missing_no_mtb["other_pathogen_hits"].apply(len) == 0),
        sum(df_missing["taxids"].isna())
    ]
})

summary_missing

,Category,Count
0,Missing AIDs (UI→Aid2Taxid),4922
1,Contain MTB TaxID,4922
2,Match Other Pathogens,0
3,Match No Known Pathogens,0
4,No TaxID Present,0


## 9 Comparing with downloaded PubChem taxid and taxonomy search

In [5]:
# Load your XML-derived filtered assays
df_xml = pd.read_csv(DATA_PROCESSED / "filtered_assays_description_results.csv")

df_xml.head()

,AID,TaxIDs_detected,ZipFolder
0,838,['1773'],0000001_0001000
1,709,['373153'],0000001_0001000
2,708,['373153'],0000001_0001000
3,557,['373153'],0000001_0001000
4,375,['1773'],0000001_0001000


In [6]:
# Parse the TaxIDs inside the XML-derived file

def parse_list(s):
    if pd.isna(s): return []
    # stored as a Python-like string: "['1773', '1234']"
    s = s.strip("[]")
    items = [x.strip(" '\"") for x in s.split(",") if x.strip()]
    return items

df_xml["Parsed_TaxIDs"] = df_xml["TaxIDs_detected"].apply(parse_list)

df_xml.head()

,AID,TaxIDs_detected,ZipFolder,Parsed_TaxIDs
0,838,['1773'],0000001_0001000,[1773]
1,709,['373153'],0000001_0001000,[373153]
2,708,['373153'],0000001_0001000,[373153]
3,557,['373153'],0000001_0001000,[373153]
4,375,['1773'],0000001_0001000,[1773]


In [7]:
# Extract AIDs explicitly linked to MTB

xml_aids = set(
    df_xml[df_xml["Parsed_TaxIDs"].apply(lambda lst: mtb_taxid in lst)]["AID"]
)

len(xml_aids)

# We already have the UI-linked AIDs as 'ui_aids'

11180

In [9]:
# Compare the 2 sets

missing_from_xml = ui_aids - xml_aids
extra_in_xml = xml_aids - ui_aids

summary = pd.DataFrame({
    "Metric": ["UI AIDs", "XML AIDs", "Missing (UI→XML)", "Unexpected (AXML→UI)"],
    "Count": [len(ui_aids), len(xml_aids), len(missing_from_xml), len(extra_in_xml)]
})

summary

,Metric,Count
0,UI AIDs,11199
1,XML AIDs,11180
2,Missing (UI→XML),19
3,Unexpected (AXML→UI),0


In [10]:
# Inspect Missing AIDs (in UI but not in xml)

df_missing_xml = df_ui[df_ui["aid"].isin(missing_from_xml)].copy()

df_missing_xml.head()

,aid,cnt,aidtype,aidname,aiddesc,aidcomment,aidsrcid,aidsrcname,aidextid,depdate,...,taxids,cellids,targettaxid,activecnt,anatomyid,anatomy,dois,pmcids,pclids,citations
10,977608,4302.0,Literature-derived,Experimentally measured binding affinity data ...,This data entry provides a collection of exper...,NaN,36,Shanghai Institute of Organic Chemistry,PDBbind-IC50 for protein-ligand complexes,20140506,...,34|210|274|287|512|536|550|562|573|615|666|727...,NaN,NaN,4302.0,NaN,NaN,10.1002/(sici)1097-0282(20000415)53:5<434::aid...,PMC11113908|PMC1157103|PMC1222325|PMC1241520|P...,159245|188532|189137|193886|208525|225116|2355...,"Hassan C, Chabrol E, Jahn L, Kester MGD, de Ru..."
11,977611,3916.0,Literature-derived,Experimentally measured binding affinity data ...,This data entry provides a collection of exper...,NaN,36,Shanghai Institute of Organic Chemistry,PDBbind-Kd for protein-ligand complexes,20140506,...,160|197|210|235|266|271|274|285|287|292|294|30...,NaN,NaN,3916.0,NaN,NaN,10.1002/(sici)1097-0134(199709)29:1<113::aid-p...,PMC1064020|PMC11114729|PMC1134746|PMC1142576|P...,152137|182245|182284|189113|208525|222463|2345...,"Shukla PK, Gautam L, Sinha M, Kaur P, Sharma S..."
12,977610,3598.0,Literature-derived,Experimentally measured binding affinity data ...,This data entry provides a collection of exper...,NaN,36,Shanghai Institute of Organic Chemistry,PDBbind-Ki for protein-ligand complexes,20140506,...,11|69|173|197|210|232|238|271|274|287|292|293|...,NaN,NaN,3598.0,NaN,NaN,10.1002/(sici)1097-0134(19990601)35:4<425::aid...,PMC10316677|PMC11114634|PMC1134741|PMC1169963|...,49317|189199|226130|233956|235576|241483|24546...,"Kick E, Martin R, Xie Y, Flatt B, Schweiger E,..."
13,1811,3073.0,Literature-derived,Experimentally measured binding affinity data ...,This data entry provides a collection of exper...,NaN,36,Shanghai Institute of Organic Chemistry,PDBBind Data 1,20090608,...,69|160|197|210|238|266|271|274|285|287|294|303...,NaN,NaN,3073.0,NaN,NaN,10.1002/(sici)1097-0134(19990601)35:4<425::aid...,PMC1064020|PMC1134741|PMC1134746|PMC1142596|PM...,49317|152137|182245|182284|188532|189113|18913...,"Smaill JB, Baker EN, Booth RJ, Bridges AJ, Dic..."
1070,1591953,24.0,Confirmatory,Inhibition of Mycobacterium tuberculosis H37Ra...,Title: Ophiobolin-Type Sesterterpenoids from t...,Journal: J Nat Prod_||_Year: 2019_||_Volume: 8...,43,ChEMBL,CHEMBL4390593,20210302,...,1773|419947,NaN,NaN,NaN,NaN,NaN,10.1021/acs.jnatprod.9b00462,NaN,17582301,"Cai R, Jiang H, Mo Y, Guo H, Li C, Long Y, Zan..."


In [11]:
df_missing_xml["aidsrcname"].value_counts()

aidsrcname
ChEMBL                                     15
Shanghai Institute of Organic Chemistry     4
Name: count, dtype: int64

In [12]:
df_missing_xml["aidtype"].value_counts()

aidtype
Literature-derived    11
Confirmatory           8
Name: count, dtype: int64

In [18]:
# Evaluate TaxIDs inside the `taxids` column
df_missing_xml["parsed_taxids"] = df_missing_xml["taxids"].apply(extract_taxids)

df_missing_xml["has_mtb"] = df_missing_xml["parsed_taxids"].apply(
    lambda lst: mtb_taxid in lst
)

df_missing_xml["has_mtb"].value_counts()

has_mtb
True    19
Name: count, dtype: int64

Compared to Aid2Taxid, we are getting ALL assays with the taxid correct! :)

In [19]:
# Inspect AIDs that DO contain the MTB TaxID
df_missing_xml_with_mtb = df_missing_xml[df_missing_xml["has_mtb"]]

df_missing_xml_with_mtb[["aid", "aidsrcname", "aidname", "taxids", "parsed_taxids"]].head(20)

,aid,aidsrcname,aidname,taxids,parsed_taxids
10,977608,Shanghai Institute of Organic Chemistry,Experimentally measured binding affinity data ...,34|210|274|287|512|536|550|562|573|615|666|727...,"[34, 210, 274, 287, 512, 536, 550, 562, 573, 6..."
11,977611,Shanghai Institute of Organic Chemistry,Experimentally measured binding affinity data ...,160|197|210|235|266|271|274|285|287|292|294|30...,"[160, 197, 210, 235, 266, 271, 274, 285, 287, ..."
12,977610,Shanghai Institute of Organic Chemistry,Experimentally measured binding affinity data ...,11|69|173|197|210|232|238|271|274|287|292|293|...,"[11, 69, 173, 197, 210, 232, 238, 271, 274, 28..."
13,1811,Shanghai Institute of Organic Chemistry,Experimentally measured binding affinity data ...,69|160|197|210|238|266|271|274|285|287|294|303...,"[69, 160, 197, 210, 238, 266, 271, 274, 285, 2..."
1070,1591953,ChEMBL,Inhibition of Mycobacterium tuberculosis H37Ra...,1773|419947,"[1773, 419947]"
1225,1385544,ChEMBL,Inhibition of Mycobacterium PTPB at 50 uM usin...,1763|1773,"[1763, 1773]"
2167,1386413,ChEMBL,Inhibition of Mycobacterium tuberculosis H37Rv...,1773|83332,"[1773, 83332]"
2276,1354805,ChEMBL,Inhibition of Mycobacterium tuberculosis H37Ra...,1773|419947,"[1773, 419947]"
3353,1541621,ChEMBL,Inhibition of Mycobacterium tuberculosis H37Rv...,1773|83332,"[1773, 83332]"
4126,1462366,ChEMBL,Antitubercular activity against Mycobacterium ...,1773|652616,"[1773, 652616]"
